In [0]:
%run ../00_common/data_utils

In [0]:
def upsert_line_unbind_records(
        itermediate_consumer_df, 
        itermediate_emedia_df, 
        unbind_records_table_name
    ):
    """
    处理UNBIND记录表的写入逻辑（所有market）。

    包含两步：
    1. 从clean consumer/emedia中提取UNBIND记录
    2. merge到t_sconsumermedia_line_unbind_records（按market+address）
    """
    # 提取所有market下的UNBIND emedia记录
    line_unbind_records = (
        itermediate_consumer_df.alias("srcc")
        .join(
            itermediate_emedia_df.alias("srce"),
            (F.col("srcc.srcc_id") == F.col("srce.srce_srcc_id")) &
            (F.col("srcc.srcc_mrkt_code") == F.col("srce.srce_mrkt_code")),
            "inner"
        )
        .where(
            F.upper(F.col("srcc.srcc_action")) == "UNBIND"
        )
        .select(
            F.col("srcc.srcc_mrkt_code"),
            F.col("srcc.srcc_id"),
            F.col("srce.srce_emdt_code"),
            F.col("srce.srce_sourcetimestamp"),
            F.col("srce.srce_address"),
            F.col("srce.srce_validitycode"),
            F.col("srce.srce_primary_flag"),
            F.col("srce.srce_appid"),
            F.col("srce.srce_contactoptinflag"),
            F.col("srce.srce_referenceemediatypecode"),
            F.col("srce.srce_referenceemediaaddress"),
            F.col("srce.srce_quality_code"),
            F.col("srce.srce_quality_desc")
        )
        .distinct()
    )

    # 标准化字段并准备upsert数据
    line_unbind_to_upsert = line_unbind_records.select(
        F.col("srcc_mrkt_code").alias("scme_market_code"),
        F.col("srcc_id").alias("scme_srcc_id"),
        F.col("srce_emdt_code").alias("scme_emdt_code"),
        F.col("srce_sourcetimestamp").alias("scme_sourcetimestamp"),
        F.col("srce_address").alias("scme_address"),
        F.col("srce_validitycode").alias("scme_validitycode"),
        F.col("srce_primary_flag").alias("scme_primary_flag"),
        F.col("srce_appid").alias("scme_appid"),
        F.col("srce_contactoptinflag").alias("scme_contactoptinflag"),
        F.col("srce_referenceemediatypecode").alias("scme_referenceemediatypecode"),
        F.col("srce_referenceemediaaddress").alias("scme_referenceemediaaddress"),
        F.col("srce_quality_code").alias("scme_quality_code"),
        F.col("srce_quality_desc").alias("scme_quality_desc"),
        F.current_timestamp().alias("scme_creation_dt"),
        F.lit("ELC").alias("scme_creation_uid"),
        F.current_timestamp().alias("scme_update_dt"),
        F.lit("ELC").alias("scme_update_uid"),
        F.lit(True).alias("scme_update_flag")
    ) \
    .withColumn("rn", F.row_number().over(Window.partitionBy("scme_market_code", "scme_address").orderBy(F.col("scme_sourcetimestamp").desc()))) \
    .filter(F.col("rn") == 1) \
    .drop("rn")

    if not line_unbind_to_upsert.isEmpty():
        unbind_records_delta = DeltaTable.forName(spark, unbind_records_table_name)
        unbind_records_delta.alias("target") \
        .merge(
            line_unbind_to_upsert.alias("source"),
            """
            target.scme_market_code = source.scme_market_code AND
            target.scme_address = source.scme_address
            """
        ) \
        .whenMatchedUpdate(
            set={
                "scme_srcc_id": F.col("source.scme_srcc_id"),
                "scme_emdt_code": F.col("source.scme_emdt_code"),
                "scme_sourcetimestamp": F.col("source.scme_sourcetimestamp"),
                "scme_validitycode": F.col("source.scme_validitycode"),
                "scme_primary_flag": F.col("source.scme_primary_flag"),
                "scme_appid": F.col("source.scme_appid"),
                "scme_contactoptinflag": F.col("source.scme_contactoptinflag"),
                "scme_referenceemediatypecode": F.col("source.scme_referenceemediatypecode"),
                "scme_referenceemediaaddress": F.col("source.scme_referenceemediaaddress"),
                "scme_quality_code": F.col("source.scme_quality_code"),
                "scme_quality_desc": F.col("source.scme_quality_desc"),
                "scme_update_dt": F.current_timestamp(),
                "scme_update_uid": F.lit("ELC"),
                "scme_update_flag": F.lit(True)
            }
        ) \
        .whenNotMatchedInsertAll() \
        .execute()
        print(f"sconsumermedia_line_unbind_records upserted: {line_unbind_to_upsert.count()} records")


def perform_line_unbind_across_profiles(
        itermediate_consumer_df,
        itermediate_emedia_df,
        master_emedia_df,
        master_emedia_table_name,
        unbind_records_table_name,
        task_id
    ):
    """
    执行跨profile的line unbind处理（所有market）。

    步骤：
    1. 从UNBIND记录表中匹配当前批次的UNBIND源记录
    2. 找出需要清空address的master emedia记录
    3. 更新master emedia地址
    4. 回写受影响master consumer的task_id
    """
    # 加载unbind records表
    unbind_records_df = spark.table(unbind_records_table_name)\
        .withColumn("rn", F.row_number().over(Window.partitionBy("scme_market_code", "scme_address").orderBy(F.col("scme_sourcetimestamp").desc()))) \
        .filter(F.col("rn") == 1) \
        .drop("rn")

    # 创建line_unbind_sconsumermedia_src临时表
    # 找到匹配的UNBIND记录（srce与lur的address和timestamp都匹配）
    line_unbind_src = (
        itermediate_emedia_df.alias("srce")
        .join(
            itermediate_consumer_df.alias("srcc"),
            (F.col("srcc.srcc_id") == F.col("srce.srce_srcc_id")) &
            (F.col("srcc.srcc_mrkt_code") == F.col("srce.srce_mrkt_code")),
            "inner"
        )
        .join(
            unbind_records_df.alias("lur"),
            (F.col("srce.srce_address") == F.col("lur.scme_address")) &
            (F.col("srcc.srcc_mrkt_code") == F.col("lur.scme_market_code")),
            "inner"
        )
        .where(
            (F.upper(F.col("srcc.srcc_action")) == "UNBIND") &
            (F.col("srce.srce_sourcetimestamp") == F.col("lur.scme_sourcetimestamp"))
        )
        .select(F.col("lur.*"))
        .distinct()
    )

    if not line_unbind_src.isEmpty():
        # 找到需要unbind的sconsumermedia记录(这些binded数据可能是之前批次的，也可能是当前批次的，但只要address相同且timestamp更早都需要处理)
        # 条件: 同样的address，但lur的timestamp更新（大于scme的timestamp）
        # 先计算UNBIND候选，并落地到临时Delta快照，避免后续merge更新触发重算
        sconsumermedia_unbind = (
            master_emedia_df.alias("scme")
            .join(
                line_unbind_src.alias("lur"),
                (F.col("lur.scme_market_code") == F.col("scme.scme_mrkt_code")) &
                (F.col("lur.scme_address") == F.col("scme.scme_address")) &
                (F.col("lur.scme_sourcetimestamp") > F.col("scme.scme_sourcetimestamp")),
                "inner"
            )
            .select(F.col("scme.*"))
        )

        if not sconsumermedia_unbind.isEmpty():
            # 将需要unbind的记录写入临时Delta快照 (避免merge更新触发重算问题)
            unbind_snapshot_path = f"dbfs:/tmp/cdp-mdm/sconsumermedia_unbind_snapshot/{task_id}"
            sconsumermedia_unbind.write.format("delta").mode("overwrite").save(unbind_snapshot_path)

            try:
                sconsumermedia_unbind_snapshot = spark.read.format("delta").load(unbind_snapshot_path)
                sconsumermedia_unbind_count = sconsumermedia_unbind_snapshot.count()

                if sconsumermedia_unbind_count > 0:
                    # 更新sconsumermedia表，将address设置为空
                    master_emedia_delta_table = DeltaTable.forName(spark, master_emedia_table_name)
                    master_emedia_delta_table.alias("target") \
                    .merge(
                        sconsumermedia_unbind_snapshot.select("scme_id", "scme_mrkt_code").distinct().alias("source"),
                        """
                        target.scme_id = source.scme_id AND
                        target.scme_mrkt_code = source.scme_mrkt_code
                        """
                    ) \
                    .whenMatchedUpdate(
                        set={
                            "scme_address": F.lit(""),
                            "scme_update_dt": F.current_timestamp(),
                            "scme_update_uid": F.lit("ELC")
                        }
                    ) \
                    .execute()
                    print(f"sconsumermedia updated (address cleared): {sconsumermedia_unbind_count} records")

                    # 更新sconsumer表task_id
                    affected_scon_ids = (
                        sconsumermedia_unbind_snapshot
                        .select("scme_scon_id", "scme_mrkt_code")
                        .distinct()
                        .cache()
                    )

                    affected_scon_ids_count = affected_scon_ids.count()

                    master_consumer_table_name = f"{get_env_config('golden_consumer_master_database')}.t_master_consumer"
                    master_consumer_delta = DeltaTable.forName(spark, master_consumer_table_name)
                    master_consumer_delta.alias("target") \
                    .merge(
                        affected_scon_ids.alias("source"),
                        """
                        target.scon_id = source.scme_scon_id AND
                        target.scon_mrkt_code = source.scme_mrkt_code
                        """
                    ) \
                    .whenMatchedUpdate(
                        set={
                            "task_id": F.lit(task_id),
                            "scon_update_dt": F.current_timestamp(),
                            "scon_update_uid": F.lit("ELC")
                        }
                    ) \
                    .execute()

                    print(f"sconsumer updated (task_id): {affected_scon_ids_count} records")

                    affected_scon_ids.unpersist()
            finally:
                try:
                    dbutils.fs.rm(unbind_snapshot_path, True)
                except Exception as e:
                    print(f"Warning: Failed to cleanup unbind snapshot path: {e}")

In [0]:
def calc_consumer_emedia_master(task_id):
    """
    计算并更新consumer emedia主数据
    支持所有Region的SQL逻辑,包括:
    1. 标准的DELETE操作处理
    2. UNBIND动作相关处理
    3. TWN Region的source system code额外过滤
    
    Args:
        task_id: 任务ID
    """
    master_emedia_table_name = f"{get_env_config('golden_consumer_master_database')}.t_master_emedia"
    # 1. 初始化数据源
    itermediate_consumer_df = spark.table(f"{get_env_config('silver_consumer_cleansed_database')}.t_clean_consumer") \
        .where(f"TASK_ID = '{task_id}'") \
        .filter(F.col("IS_INCLUDE") == True) 

    itermediate_emedia_df = spark.table(f"{get_env_config('silver_consumer_cleansed_database')}.t_clean_emedia") \
        .where(f"TASK_ID = '{task_id}'") 

    master_consumer_df = spark.table(f"{get_env_config('golden_consumer_master_database')}.t_master_consumer")
    master_emedia_df = spark.table(master_emedia_table_name)

    # UNBIND记录表: sconsumermedia_line_unbind_records
    unbind_records_table_name = f"{get_env_config('golden_consumer_master_database')}.t_sconsumermedia_line_unbind_records"
    unbind_records_df = spark.table(unbind_records_table_name) \
        .withColumn("rn", F.row_number().over(Window.partitionBy("scme_market_code", "scme_address").orderBy(F.col("scme_sourcetimestamp").desc()))) \
        .filter(F.col("rn") == 1) \
        .drop("rn")

    # TWN Region特殊表: tsourcesystemcodesurvivelatesttimestamp
    source_system_survive_df = spark.table(f"{get_env_config('golden_consumer_master_database')}.t_sourcesystemcode_survive")
    
    # 2. 查询query_emedia
    # 构建基础DataFrame - 包含所有必要的join
    base_emedia_df = (
        itermediate_consumer_df.alias("srcc")
        .join(
            itermediate_emedia_df.alias("srce"),
            (F.col("srcc.srcc_id") == F.col("srce.srce_srcc_id")) &
            (F.col("srcc.srcc_mrkt_code") == F.col("srce.srce_mrkt_code")),
            "inner"
        )
        .join(
            master_consumer_df.alias("scon"),
            (F.col("srce.srce_srcc_id") == F.col("scon.scon_srcc_id")) &
            (F.col("srce.srce_mrkt_code") == F.col("scon.scon_mrkt_code")),
            "inner"
        )
        .join(
            master_emedia_df.alias("scme"),
            (F.col("scon.scon_id") == F.col("scme.scme_scon_id")) &
            (F.col("scon.scon_mrkt_code") == F.col("scme.scme_mrkt_code")) & 
            (F.col("srce.srce_emdt_code") == F.col("scme.scme_emdt_code")),
            "left"
        )
        .join(
            unbind_records_df.alias("lur"),
            (F.col("srce.srce_address") == F.col("lur.scme_address")) &
            (F.col("srcc.srcc_mrkt_code") == F.col("lur.scme_market_code")),
            "left"
        )
        .join(
            source_system_survive_df.alias("sss"),
            (F.col("srcc.srcc_srcs_code") == F.col("sss.srcs_code")) &
            (F.col("srcc.srcc_mrkt_code") == F.col("sss.market_code")),
            "left"
        )
    )

    # 构建where条件组合
    # 基础条件: address或quality_code不同
    base_where_condition = (
        (F.coalesce(F.col("srce.srce_address"), F.lit("")) != F.coalesce(F.col("scme.scme_address"), F.lit(""))) |
        (F.coalesce(F.col("srce.srce_quality_code"), F.lit("")) != F.coalesce(F.col("scme.scme_quality_code"), F.lit("inv")))
    )
    
    # TWN Region额外条件: srcc_srcs_code在白名单表中（通过left join后判断非null）
    twn_condition = F.col("sss.srcs_code").isNotNull()
    
    # # (re)Binding额外条件: sconsumermedia_line_unbind_records过滤逻辑（所有market）
    # # Case 1: Address不在记录表中 (lur.scme_address is null) - 包含
    # # Case 2: Address在记录表中但incoming timestamp更新 - 包含
    # # Case 3: Address在记录表中且incoming timestamp更旧 - 排除
    # binding_exclusion_condition = (
    #     F.col("lur.scme_address").isNull() |  # 不在unbind记录表中
    #     (
    #         F.col("lur.scme_address").isNotNull() &  # 在unbind记录表中且incoming timestamp更新
    #         (F.col("srce.srce_sourcetimestamp") > F.col("lur.scme_sourcetimestamp"))
    #     )
    # )

    # 当rebinding中的address在unbind记录表中存在且incoming timestamp更旧，需要清空当前批次的address值
    rebinding_address_empty_flag = (
        F.col("lur.scme_address").isNotNull() & 
            (F.col("srce.srce_sourcetimestamp") < F.col("lur.scme_sourcetimestamp"))
    )
    
    # 组合所有条件
    combined_where_condition = (base_where_condition | twn_condition)
    
    # 应用where条件并选择字段
    master_emedia_to_insert = (
        base_emedia_df
        .where(combined_where_condition)
        .select(
            F.col("scon.consumermdmkey"),
            F.col("scon.scon_id"),
            F.col("srce.srce_mrkt_code"),
            F.col("srce.srce_emdt_code"),
            # 仅DELETE时使用srcc_sourcetimestamp且address置空，其他情况使用srce的timestamp和address
            F.when(F.upper(F.col("srcc.srcc_action")) == "DELETE", F.col("srcc.srcc_sourcetimestamp")).otherwise(F.col("srce.srce_sourcetimestamp")).alias("srce_sourcetimestamp"),
            F.when(
                (F.upper(F.col("srcc.srcc_action")) == "DELETE") | rebinding_address_empty_flag,
                F.lit("")
            ).otherwise(F.col("srce.srce_address")).alias("srce_address"),
            F.col("srce.srce_validitycode"),
            F.col("srce.srce_primary_flag"),
            F.col("srce.srce_appid"),
            F.col("srce.srce_contactoptinflag"),
            F.col("srce.srce_referenceemediatypecode"),
            F.col("srce.srce_referenceemediaaddress"),
            F.col("srce.srce_quality_code"),
            F.col("srce.srce_quality_desc"),
            F.col("srce.batch_id")
        )
        .distinct()
        .drop("consumermdmkey")
        .select(
            F.expr("uuid()").alias("scme_id"),
            F.col("scon_id").alias("scme_scon_id"),
            F.col("srce_mrkt_code").alias("scme_mrkt_code"),
            F.col("srce_emdt_code").alias("scme_emdt_code"),
            F.col("srce_sourcetimestamp").alias("scme_sourcetimestamp"),
            F.col("srce_address").alias("scme_address"),
            F.col("srce_validitycode").alias("scme_validitycode"),
            F.col("srce_primary_flag").alias("scme_primary_flag"),
            F.col("srce_appid").alias("scme_appid"),
            F.col("srce_contactoptinflag").alias("scme_contactoptinflag"),
            F.col("srce_referenceemediatypecode").alias("scme_referenceemediatypecode"),
            F.col("srce_referenceemediaaddress").alias("scme_referenceemediaaddress"),
            F.col("srce_quality_code").alias("scme_quality_code"),
            F.col("srce_quality_desc").alias("scme_quality_desc"),
            F.current_timestamp().alias("scme_creation_dt"),
            F.lit("ELC").alias("scme_creation_uid"),
            F.current_timestamp().alias("scme_update_dt"),
            F.lit("ELC").alias("scme_update_uid"),
            F.lit(True).alias("scme_update_flag"),
            F.col("batch_id"),
            F.lit(task_id).alias("task_id")
        )
    )
    master_emedia_to_insert = master_emedia_to_insert.checkpoint(eager=True)
    master_emedia_insert_count = master_emedia_to_insert.count()

    # 3. 执行query_emedia_delete标准删除逻辑（所有Region）
    master_emedia_delete_count = 0
    # 构建基础删除查询
    standard_delete_base = (
        itermediate_emedia_df.alias("srce")
        .join(
            master_consumer_df.alias("scon"),
            (F.col("srce.srce_srcc_id") == F.col("scon.scon_srcc_id")) &
            (F.col("srce.srce_mrkt_code") == F.col("scon.scon_mrkt_code")),
            "inner"
        )
        .join(
            master_emedia_df.alias("scme"),
            (F.col("scon.scon_id") == F.col("scme.scme_scon_id")) &
            (F.col("scon.scon_mrkt_code") == F.col("scme.scme_mrkt_code")) & 
            (F.col("srce.srce_emdt_code") == F.col("scme.scme_emdt_code")),
            "inner"
        )
        # TWN Region需要额外join ssourceconsumer表和白名单表
        .join(
            itermediate_consumer_df.alias("srcc"),
            (F.col("srce.srce_srcc_id") == F.col("srcc.srcc_id")) &
            (F.col("srce.srce_mrkt_code") == F.col("srcc.srcc_mrkt_code")),
            "left"  # left join以保留非TWN的记录
        )
        .join(
            source_system_survive_df.alias("sss"),
            (F.col("srcc.srcc_srcs_code") == F.col("sss.srcs_code")) &
            (F.col("srcc.srcc_mrkt_code") == F.col("sss.market_code")),
            "left"
        )
    )
    
    # 构建where条件
    # 基础条件: address或quality_code不同
    base_delete_condition = (
        (F.coalesce(F.col("srce.srce_address"), F.lit("")) != F.coalesce(F.col("scme.scme_address"), F.lit(""))) |
        (F.coalesce(F.col("srce.srce_quality_code"), F.lit("")) != F.coalesce(F.col("scme.scme_quality_code"), F.lit("inv")))
    )
    
    # TWN额外条件: srcc_srcs_code在白名单中
    twn_delete_condition = F.col("sss.srcs_code").isNotNull()
    
    # 组合条件: 基础条件 OR TWN条件
    combined_delete_condition = (base_delete_condition | twn_delete_condition)
    
    master_emedia_to_delete = (
        standard_delete_base
        .where(combined_delete_condition)
        .select(F.col("scme.*")).distinct()
    )
    
    # 执行标准删除
    if not master_emedia_to_delete.isEmpty():
        master_emedia_delta_table = DeltaTable.forName(spark, master_emedia_table_name)
        master_emedia_merge_result = (
            master_emedia_delta_table
                .alias("t")
                .merge(
                    master_emedia_to_delete.alias("s"),
                    """
                    t.scme_id = s.scme_id AND
                    t.scme_mrkt_code = s.scme_mrkt_code
                    """
                )
                .whenMatchedDelete()
                .execute()
        )
        # master_emedia_delete_count = master_emedia_merge_result.first().num_deleted_rows

    # 4. 插入数据到目标主表
    if master_emedia_insert_count > 0:
        append_table(master_emedia_to_insert, master_emedia_table_name)
    print(f'master emedia inserted count: {master_emedia_insert_count}, deleted count: {master_emedia_delete_count}')

    # 5. UNBIND处理
    # 5.1 sconsumermedia_line_unbind_records表的处理（兼容所有market）
    upsert_line_unbind_records(
        itermediate_consumer_df, 
        itermediate_emedia_df, 
        unbind_records_table_name
    )
    
    # 5.2 执行sp_PerformLineUnbindAcrossProfiles存储过程逻辑（适用于所有market）
    perform_line_unbind_across_profiles(
        itermediate_consumer_df,
        itermediate_emedia_df,
        master_emedia_df,
        master_emedia_table_name,
        unbind_records_table_name,
        task_id
    )
            

In [0]:
task_id = dbutils.widgets.get("task_id")
print(f"task_id: {task_id}")

with StepLogger("5.2_generate_master_emedia_tables", "05-2", "consumerlist", task_id=task_id) as logger:
    spark.sparkContext.setCheckpointDir(f"{get_env_config('checkpoint_path_consumer_master')}/{task_id}")
    calc_consumer_emedia_master(task_id)